<a href="https://colab.research.google.com/github/nuhuynhh/AAI2026/blob/main/05_refund_policy.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Setup Models

In [1]:
!pip install -U langchain-google-genai langgraph langchain langchain-community pandas

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 168.1/168.1 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 112.5/112.5 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 39.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 87.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 44.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.0/65.0 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 3.3 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
  Attempting uninstall: pandas
    Found existing installation: pandas 2.2.2
    Uninstalling pandas-2.2.2:
      Successfully uni

In [3]:
from langchain_google_genai import ChatGoogleGenerativeAI
import os
from google.colab import userdata


google_key = userdata.get("GOOGLE_API_KEY")
if not google_key:
    raise ValueError("GOOGLE_API_KEY secret is not available. Enable it in Colab Secrets.")

os.environ["GOOGLE_API_KEY"] = google_key
model = ChatGoogleGenerativeAI(
    model="gemini-2.0-flash",
    temperature=0
)

In [4]:
from langchain_google_genai import ChatGoogleGenerativeAI

model = ChatGoogleGenerativeAI(
    model="gemini-2.0-flash",
    temperature=0
)

## 05.02. Define refund and return policy tools

In [5]:
from langchain_core.tools import tool

REFUND_POLICY_TEXT = """
Laptop Store Refund and Return Policy

1. Standard returns:
- Customers may return most laptops within 30 days of delivery.
- The item must be returned with the charger and original accessories.

2. Condition rules:
- Unopened items qualify for a full refund.
- Opened items in like-new condition qualify for a full refund within 30 days.
- Items with customer-caused damage are not eligible for a standard refund.
- Manufacturer defects are eligible for replacement, repair, or refund after inspection.

3. Shipping and fees:
- Return shipping is free for defective or incorrect items.
- For non-defective returns, a 10 percent restocking fee may apply to opened laptops.
- Original shipping fees are non-refundable unless the seller made the mistake.

4. Exceptions:
- Final-sale items are not returnable.
- Gift cards and software activation keys are not refundable.
- Returns requested after 30 days are normally denied unless required by law or approved by support.

5. Refund processing:
- Approved refunds are sent to the original payment method.
- Refunds usually appear within 5 to 10 business days after inspection.

6. Order help:
- Customers should provide an order ID when asking about order-specific refund eligibility.
"""

@tool
def get_refund_policy(topic: str) -> str:
    """Return the store's refund and return policy details for a requested topic."""
    topic_lower = topic.lower().strip()

    if any(word in topic_lower for word in ["fee", "restocking", "shipping"]):
        return "Return shipping is free for defective or incorrect items. For non-defective opened laptop returns, a 10 percent restocking fee may apply. Original shipping fees are non-refundable unless the seller made the mistake."

    if any(word in topic_lower for word in ["timeline", "days", "window", "30", "deadline"]):
        return "Most laptops can be returned within 30 days of delivery. Requests after 30 days are normally denied unless required by law or specially approved by support."

    if any(word in topic_lower for word in ["defect", "broken", "damaged", "replacement"]):
        return "Manufacturer defects can qualify for replacement, repair, or refund after inspection. Customer-caused damage is not eligible for a standard refund."

    if any(word in topic_lower for word in ["process", "payment", "refund method", "how long", "business days"]):
        return "Approved refunds go back to the original payment method and usually appear within 5 to 10 business days after inspection."

    if any(word in topic_lower for word in ["final sale", "gift card", "software", "exception"]):
        return "Final-sale items, gift cards, and software activation keys are not refundable."

    return REFUND_POLICY_TEXT

@tool
def check_return_eligibility(days_since_delivery: int, opened: bool, damaged: bool = False, defective: bool = False, final_sale: bool = False) -> str:
    """Check whether an item is likely eligible for a return or refund based on store policy."""
    if final_sale:
        return "Not eligible: final-sale items cannot be returned or refunded."
    if days_since_delivery > 30 and not defective:
        return "Not eligible under the standard policy: most returns must be requested within 30 days of delivery."
    if damaged and not defective:
        return "Not eligible for a standard refund: customer-caused damage is excluded from the standard return policy."
    if defective:
        return "Likely eligible: defective items may qualify for repair, replacement, or refund after inspection."
    if opened:
        return "Likely eligible: opened items in like-new condition can be returned within 30 days, though a 10 percent restocking fee may apply."
    return "Eligible: unopened items returned within 30 days qualify for a full refund."

@tool
def get_policy_summary() -> str:
    """Return a concise summary of the store's refund and return policy."""
    return (
        "Most laptops can be returned within 30 days of delivery. "
        "Opened items in like-new condition are usually eligible, though a 10 percent restocking fee may apply. "
        "Defective items may qualify for repair, replacement, or refund. "
        "Final-sale items, gift cards, and software activation keys are not refundable. "
        "Approved refunds usually appear within 5 to 10 business days after inspection."
    )
